In [ ]:
import ROOT
%load_ext JupyROOT

In [ ]:
%%cpp
const long long NENTRIES = 100;
const long long NEXPERIMENTS = 100000;

const double mu = 10.;
const double sigma = 2.;
bool isOneSided = false;
const double zobs = 2.; //zobs > 0, it only works in intervals of 0.1 

In [ ]:
%%cpp
    gStyle->SetOptStat(0);
    TRandom3 *rand = new TRandom3(0);

    TH1F *Zvalues = new TH1F("Zvalues", ";Z;Entries", 100, -5., 5.);
    TH1F *Xvalues = new TH1F("Xvalues", ";x;Entries", 100, 0., 20.);

    for(int i = 0; i < NEXPERIMENTS; i++)
    {
        //Welford's variables
        long long N = 0;
        double mean = 0.0;
        double M2 = 0.0;

        for(int j = 0; j < NENTRIES; j++)
        {
            double u1 = rand->Uniform(0., 1.);
            double u2 = rand->Uniform(0., 1.);

            while(u1 == 0) //Avoid ln(0)
            {
                u2 = rand->Uniform(0., 1.);
            }

            //A buffer could be implemented so that we don't discard z_gen = sqrt(-2.*log(u1))*sin(2.*TMath::Pi()*u2). For the future...
            double z_gen = sqrt(-2.*log(u1))*cos(2.*TMath::Pi()*u2);
            
            double x = mu + sigma*z_gen;
            Xvalues->Fill(x);

            //Apply Welford's updates
            N++;
            double delta = x - mean;
            mean += delta / N;
            double delta2 = x - mean;
            M2 += delta * delta2;
        }
        double variance = (N > 1) ? M2 / (N - 1) : 0.0; //Use (N - 1) for sample variance

        double z = (mean - mu)/(sqrt(variance/NENTRIES));

        Zvalues->Fill(z);

    }

    double pvalue = 1.-Xvalues->Integral(0, 50+int(zobs/0.1))/(NENTRIES*NEXPERIMENTS);
    std::string spvalue =  Form("MC p-value(Z>= %.1lf) = %.07lf", zobs, pvalue);

    if(!isOneSided)
    {
        pvalue *= 2.;
        spvalue =  Form("MC p-value(|Z|>= %.1lf) = %.07lf", zobs, pvalue);
    }

    std::cout << spvalue << std::endl;

    TCanvas *cValues = new TCanvas("cValues", "cValues", 1100, 550);
    cValues->Divide(2,1);
    cValues->cd(1);
    gPad->SetLeftMargin(0.15);
    Xvalues->SetLineWidth(2);
    Xvalues->SetLineColor(kBlue);
    Xvalues->Draw();
    cValues->cd(2);
    gPad->SetLeftMargin(0.15);
    Zvalues->SetLineWidth(2);
    Zvalues->SetLineColor(kRed);
    Zvalues->Draw();
    cValues->Draw();